In [18]:
import os
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# ── Paths ──────────────────────────────────────────────────────────────────
# Switch this depending on environment
KAGGLE = True

if KAGGLE:
    DATA_ROOT = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
else:
    DATA_ROOT = r"C:\\Users\\fmatt\\OneDrive\\Desktop\\AI_lab\\deepfake-detection\\data"

# ── Device ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Data root: {DATA_ROOT}")

Using device: cuda
Data root: /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23


In [ ]:
names = [item for item in os.listdir("/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/")]

PREPROCESSING PIPELINE

In [ ]:
# define real and fake videos dirs
real_dir = os.path.join(DATA_ROOT, "original")
fake_dirs = [os.path.join(DATA_ROOT, name) for name in names if name!="original" and name!= "csv"]

In [ ]:
fake_dirs, real_dir

In [ ]:
real_vids = [(os.path.join(real_dir, filename), "real") for filename in os.listdir(real_dir)]
# fake_vids = [(os.path.join(fake_dir, filename), "fake") for fake_dir in fake_dirs for filename in os.listdir(fake_dir)]   USELESS

In [ ]:
len(real_vids)

In [ ]:
# Split real videos into train/val/test (80/10/10) 
# at video level to prevent data leakage
from sklearn.model_selection import train_test_split

train_real, others = train_test_split(real_vids, test_size=0.2, random_state=42)
val_real, test_real = train_test_split(others, test_size=0.5, random_state=42)

train_fake, val_fake, test_fake = [], [], []

for fake_dir in fake_dirs:
    fake_vids = [(os.path.join(fake_dir, filename), "fake") for filename in os.listdir(fake_dir)]
    train, others = train_test_split(fake_vids, test_size=0.2, random_state=42)
    val, test= train_test_split(others, test_size=0.5, random_state=42)
    train_fake.extend(train)
    val_fake.extend(val)
    test_fake.extend(test)

In [ ]:
len(train_real), len(train_fake), len(val_real), len(val_fake), len(test_real), len(test_fake)

In [ ]:
train_split = train_real + train_fake
val_split = val_real + val_fake
test_split = test_real + test_fake

In [ ]:
len(train_split), len(val_split), len(test_split)

In [ ]:
# Create folder structure that will contain the frames
# It must match the structure in vscode
from itertools import product

splits = ["train", "test", "val"]
classes= ["real", "fake"]
paths = [os.path.join("/kaggle/working/processed/", split, cls) for split, cls in product(splits, classes)]

# filling folders with 5 dummy images
for path in paths:
    os.makedirs(path, exist_ok=True)

In [ ]:
for item in os.listdir("/kaggle/working/processed/"):
    print(item)

In [ ]:
!pip install facenet-pytorch==2.5.2

In [ ]:
# Analyze video frames (10 frames per video)
# Detect faces and store them in the correct file
from facenet_pytorch import MTCNN
mtcnn = MTCNN(keep_all=False, device=device)

Open the video with OpenCV

Get the total number of frames

Sample frame indices uniformly (we decided 10 frames per 
video)

For each sampled index, read that frame

Run MTCNN on the frame to detect and crop the face

Save the crop to the correct output folder

In [ ]:
import cv2
import numpy as np
import PIL
def extract_faces(video_path, output_dir,num_frames=10):
    # open video
    cap = cv2.VideoCapture(video_path)
    # get the total number of frames
    frames_tot = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    # select uniformly 10 frames indeces
    frame_idxs = np.linspace(0, frames_tot-1, num_frames, dtype=int)
    for idx in frame_idxs:
        # set the position of the frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        # read the frame; it returns (succes, frame)
        _, frame = cap.read()
        # OpenCV reads BGR format while MTCNN wants RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # store cropped face; return face as pytorch tensor
        # with values in range [-1,1] (normalized internally)
        cropped_face = mtcnn(frame_rgb) 
        if cropped_face is not None:
            # unnormalize and convert to type uint8
            unnorm_face = ((cropped_face + 1)/2*255).to(torch.uint8)
            # (C,H,W) -> (H,W,C); tensor lives on the GPU while np array
            # works with CPU
            unnorm_face = unnorm_face.permute(1,2,0).cpu().numpy()
            img = PIL.Image.fromarray(unnorm_face)
            img_name = f"{os.path.basename(video_path)}_{idx}.jpg"
            img.save(os.path.join(output_dir,img_name))
    cap.release()

In [ ]:
# loop that calls the function on every video of every split
split_data = {
    "train": train_split,
    "val": val_split,
    "test": test_split
    
}

for split in split_data.keys():
    print(f"Processing {split} split...")
    for video in split_data[split]:
        video_path, cls = video
        output_dir = os.path.join("/kaggle/working/processed/", split, cls)
        extract_faces(video_path, output_dir)

In [14]:
# check where train,test,val files live
import os
for dirname, dirs, files in os.walk("/kaggle/input"):
    print(dirname)


/kaggle/input
/kaggle/input/notebooks
/kaggle/input/notebooks/matteofioretti
/kaggle/input/datasets
/kaggle/input/datasets/xdxd003
/kaggle/input/datasets/xdxd003/ff-c23
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Face2Face
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/DeepFakeDetection
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/original
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/NeuralTextures
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/FaceShifter
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/FaceSwap
/kaggle/input/datasets/matteofioretti
/kaggle/input/datasets/matteofioretti/deepfake-processed
/kaggle/input/datasets/matteofioretti/deepfake-processed/processed
/kaggle/input/datasets/matteofioretti/deepfake-processed/pro

In [15]:
count = sum(len(files) for _, _, files in os.walk("/kaggle/input/datasets/matteofioretti/deepfake-processed/processed"))
print(f"Total face crops: {count}")

Total face crops: 38723


In [19]:
DATA_ROOT = "/kaggle/input/datasets/matteofioretti/deepfake-processed/processed"

In [20]:
for split in ["train", "val", "test"]:
    for cls in ["real", "fake"]:
        path = os.path.join(DATA_ROOT, split, cls)
        count = len(os.listdir(path))
        print(f"{split}/{cls}: {count}")

train/real: 7999
train/fake: 22982
val/real: 1000
val/fake: 2886
test/real: 1000
test/fake: 2856


There is a class imbalance in the dataset, roughly 3:1 fake:ratio, therefore I will use weighted loss

In [23]:
num_real = 7999
num_fake = 22982
total_samples = num_real + num_fake

weight_real = total_samples / (2*num_real)
weight_fake = total_samples / (2*num_fake)

In [24]:
weight_real, weight_fake

(1.9365545693211652, 0.6740274997824385)

In [25]:
weight = torch.tensor([weight_fake, weight_real]).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weight)